In [1]:
from http.client import responses
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.chains.combine_documents  import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
import os
import ollama
from oauthlib.uri_validate import query
from sqlalchemy.orm.collections import collection

In [2]:
path = ("C:\Ary\langchain\cv-ahmad-bukhari.pdf")

loader = PyPDFLoader(path)
docs = loader.load()

docs

<>:1: SyntaxWarning: invalid escape sequence '\A'
<>:1: SyntaxWarning: invalid escape sequence '\A'
C:\Users\Ahmad Bukhari\AppData\Local\Temp\ipykernel_8128\4065798217.py:1: SyntaxWarning: invalid escape sequence '\A'
  path = ("C:\Ary\langchain\cv-ahmad-bukhari.pdf")


[Document(metadata={'producer': 'Skia/PDF m141', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36', 'creationdate': '2025-10-29T12:04:32+00:00', 'title': 'cv-ahmad-bukhari', 'moddate': '2025-10-29T12:04:32+00:00', 'source': 'C:\\Ary\\langchain\\cv-ahmad-bukhari.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Ahmad Bukhari\nLulusan Teknik Informatika yang memiliki daya tarik dalam bidang Data Science, Machine\nLearning, Deep Learning, dan Ai Engineer. Memiliki pengalaman dalam membangun model\nmachine learning dan deep learning, serta memiliki sifat tanggung jawab, kerjasama tim yang\nsolid, dan mudah beradaptasi\n (+62) 82394183779 |  ahmadbukhari100703@gmail.com |  Ahmad Bukhari |  arydotme |  Portfolio\nEducation\nUniversitas Teknologi Akba Makassar Makassar, Sulawesi Selatan\nTeknik Informatika (3.94) 08/2021 - 07/2025\nUniversitas Negeri Malang Malang, Jawa Timur\nTeknik Informatika (3.9

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
documents = text_splitter.split_documents(docs)

print(len(documents))

7


In [4]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text:v1.5"
)

In [5]:
db = Chroma.from_documents(documents, embedding=embeddings)

In [6]:
db

In [7]:
llm = ChatOllama(model="llama3.2:1b")

C:\Users\Ahmad Bukhari\AppData\Local\Temp\ipykernel_8128\4235846879.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="llama3.2:1b")


In [8]:
prompt = ChatPromptTemplate.from_template('''
Answer the following question based only on the provided context.
Think step by step before providing a detailed answer.
<context>
{context}
</context>
Question: {input}
''')

In [9]:
document_chain = create_stuff_documents_chain(llm, prompt)

In [10]:
"""
Retrievers: A retriever is an interface that returns documents given
 an unstructured query. It is more general than a vector store.
 A retriever does not need to be able to store documents, only to
 return (or retrieve) them. Vector stores can be used as the backbone
 of a retriever, but there are other types of retrievers as well.
 https://python.langchain.com/docs/modules/data_connection/retrievers/
"""

retriever = db.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000277AA30A7B0>, search_kwargs={})

In [11]:
"""
Retrieval chain:This chain takes in a user inquiry, which is then
passed to the retriever to fetch relevant documents. Those documents
(and original inputs) are then passed to an LLM to generate a response
https://python.langchain.com/docs/modules/chains/
"""

retriever_chain = create_retrieval_chain(retriever, document_chain)

In [12]:
response = retriever_chain.invoke({"input": "Extract email, phone number, university, and skill"})

In [13]:
response['answer']

'Based on the provided context, I can extract the following information:\n\n- Email: ahmadbukhari100703@gmail.com\n- Phone Number: +62 82394183779\n- University: Universitas Teknologi Akba Makassar Makassar, Sulawesi Selatan\n- Skill: Machine Learning Engineer'